In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import figurefirst as fifi

plt.rcParams['font.serif'] = ['Times'] + plt.rcParams['font.serif']
plt.rcParams['text.usetex'] = False
plt.rcParams["ps.usedistiller"] = 'xpdf'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams["mathtext.fontset"] = 'cm'


In [2]:
df = pd.read_parquet('df_all_species.parquet')

In [3]:
TRANSLATION = True

FIGURE_NAME = 'unifying_analysis.svg'

if TRANSLATION:
    FIGURE_NAME = 'unifying_analysis_translationTrue.svg'

# Slope bar plots

In [4]:
albatross= df[df.species =='albatross'].copy().sort_values('slope')
albatross.loc[3,'flow_condition'] = 'unknown1'
moth= df[df.species =='moths'].sort_values('slope')
shark= df[df.objid.isin(['shark_a', 'shark_d'])].sort_values('slope')



In [5]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os
from typing import Optional
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from braid_analysis import braid_analysis_plots

def format_ticks_as_latex(ax, axis='both'):
    """
    Update tick labels on a matplotlib axis to be wrapped in dollar signs
    for LaTeX rendering.
    
    Parameters:
        ax:     A matplotlib Axes object.
        axis:   Which axis to format: 'x', 'y', or 'both' (default).
    """
    def to_latex(label_text):
        text = label_text.strip()
        if text and not (text.startswith('$') and text.endswith('$')):
            return f'${text}$'
        return text

    def apply_latex_labels(mpl_axis):
        # Draw the figure to ensure tick labels are populated
        ax.figure.canvas.draw()
        labels = [tick.get_text() for tick in mpl_axis.get_ticklabels()]
        new_labels = [to_latex(l) for l in labels]
        mpl_axis.set_ticklabels(new_labels)

    if axis in ('x', 'both'):
        apply_latex_labels(ax.xaxis)
    if axis in ('y', 'both'):
        apply_latex_labels(ax.yaxis)

def to_sentence_case(text: str) -> str:
    if not text:
        return text
    return text[0].upper() + text[1:].lower()

def find_file(directory: str, str1: str, str2: str) -> Optional[str]:
    for filename in os.listdir(directory):
        if str1 in filename and str2 in filename:
            return os.path.join(directory, filename)
    return None

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import cdist

from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union
import numpy as np
from sklearn.preprocessing import MinMaxScaler

from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union

def plot_arrowhead_trajectory_scaled(x, y, color='black', arrow_length=0.05, arrow_angle=30,
                                     ax=None, linewidth=1, scale_bar=False, units='',
                                     flow_direction=None, fontsize=5,
                                     flow_arrow_length=0.05, flow_arrow_angle=30,
                                     flow_arrow_size=0.08, padding=0.2,
                                     flow_column_width=0.2):

    has_flow = flow_direction is not None and flow_arrow_length is not None

    # Get the physical size of the axis in inches to compute aspect-equal coordinate ranges
    ax.figure.canvas.draw()
    bbox = ax.get_window_extent().transformed(ax.figure.dpi_scale_trans.inverted())
    ax_width_in  = bbox.width   # physical width in inches
    ax_height_in = bbox.height  # physical height in inches

    # With set_aspect('equal'), the coordinate range ratio must match the physical ratio.
    # We choose the coordinate space to be [0, ax_width_in] x [0, ax_height_in]
    # (i.e. 1 unit = 1 inch), which naturally satisfies aspect='equal'.
    coord_width  = ax_width_in
    coord_height = ax_height_in

    # Reserve right column for flow arrow
    flow_col = flow_column_width * coord_width if has_flow else 0.0

    # Left region for trajectory and scale bar (in coordinate space)
    left_x_min = padding * coord_width
    left_x_max = coord_width - flow_col - padding * coord_width
    y_min_pad  = padding * coord_height
    y_max_pad  = coord_height - padding * coord_height

    # Scale trajectory into the left region
    x_norm = (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x))
    y_norm = (y - np.nanmin(y)) / (np.nanmax(y) - np.nanmin(y))
    x_scaled = x_norm * (left_x_max - left_x_min) + left_x_min
    y_scaled = y_norm * (y_max_pad  - y_min_pad)  + y_min_pad

    # Set axis limits explicitly to our coordinate space before plotting
    ax.set_xlim(0, coord_width)
    ax.set_ylim(0, coord_height)
    ax.set_aspect('equal')

    braid_analysis_plots.plot_arrowhead_trajectory(x_scaled, y_scaled, color=color, arrow_length=arrow_length,
                               arrow_angle=arrow_angle, ax=ax, linewidth=linewidth)

    for collection in ax.collections:
        collection.set_clip_on(False)

    # Reread limits in case set_aspect nudged them
    ax.figure.canvas.draw()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_extent_ax = xlim[1] - xlim[0]
    y_extent_ax = ylim[1] - ylim[0]

    label_gap = 0.03 * y_extent_ax
    offset_x  = 0.02 * x_extent_ax

    # Right column bounds
    right_x_min = coord_width - flow_col
    right_x_max = coord_width

    # Occupied geometry: convex hull of trajectory
    traj_hull = MultiPoint(list(zip(x_scaled, y_scaled))).convex_hull.buffer(offset_x * 2)
    occupied_geom = traj_hull

    def find_best_position(size, occupied, x_min, x_max, y_min, y_max):
        n = 30
        xs = np.linspace(x_min + size, x_max - size, n)
        ys = np.linspace(y_min + size, y_max - size, n)
        best_pt, best_dist = None, -1
        for cx_ in xs:
            for cy_ in ys:
                pt = Point(cx_, cy_)
                d = pt.distance(occupied) if not occupied.is_empty else 1e9
                box_fits = (cx_ - size > x_min and cx_ + size < x_max and
                            cy_ - size > y_min and cy_ + size < y_max)
                if box_fits and d > best_dist:
                    best_dist = d
                    best_pt = (cx_, cy_)
        return best_pt

    # --- Scale bar ---
    if scale_bar:
        x_extent_original = np.nanmax(x) - np.nanmin(x)
        max_bar_original = 0.3 * x_extent_original
        magnitude = 10 ** np.floor(np.log10(max_bar_original))
        nice_steps = [1, 2, 5]
        bar_size_original = magnitude
        for step in nice_steps:
            candidate = step * magnitude
            if candidate <= max_bar_original:
                bar_size_original = candidate

        scale_factor = (left_x_max - left_x_min) / x_extent_original
        bar_size_scaled = bar_size_original * scale_factor
        bar_half = bar_size_scaled / 2

        pos = find_best_position(max(bar_half, label_gap * 2), occupied_geom,
                                 xlim[0], right_x_min, ylim[0], ylim[1])
        if pos is not None:
            cx, cy = pos
            bar_x_start = cx - bar_half
            bar_x_end   = cx + bar_half

            if cy < (ylim[0] + ylim[1]) / 2:
                text_y, va = cy + label_gap, 'bottom'
            else:
                text_y, va = cy - label_gap, 'top'

            ax.plot([bar_x_start, bar_x_end], [cy, cy], color=color, linewidth=0.5)
            ax.text(cx, text_y, f'{bar_size_original:g} {units}',
                    ha='center', va=va, fontsize=fontsize, color=color)

            bar_geom = MultiPoint([
                (bar_x_start, cy), (bar_x_end, cy), (cx, text_y)
            ]).convex_hull.buffer(label_gap * 2)
            occupied_geom = unary_union([occupied_geom, bar_geom])

    # --- Flow direction arrow ---
    if has_flow:
        arrow_size_scaled = flow_arrow_size * x_extent_ax

        cx = (right_x_min + right_x_max) / 2
        cy = (ylim[0] + ylim[1]) / 2

        dx = np.cos(flow_direction) * arrow_size_scaled / 2
        dy = np.sin(flow_direction) * arrow_size_scaled / 2

        n_points = 10
        t = np.linspace(-0.5, 0.5, n_points)
        arrow_x = cx + t * dx * 2
        arrow_y = cy + t * dy * 2

        braid_analysis_plots.plot_arrowhead_trajectory(
            arrow_x, arrow_y,
            color=color,
            arrow_length=flow_arrow_length,
            arrow_angle=flow_arrow_angle,
            ax=ax,
            linewidth=linewidth
        )

        for collection in ax.collections:
            collection.set_clip_on(False)

        text_angle_deg = np.degrees(flow_direction)
        if 90 < text_angle_deg % 360 < 270:
            text_angle_deg += 180

        perp_dx = -np.sin(flow_direction) * label_gap * 3
        perp_dy =  np.cos(flow_direction) * label_gap * 3
        cx_mid = (right_x_min + right_x_max) / 2
        cy_mid = (ylim[0] + ylim[1]) / 2
        if (cx + perp_dx - cx_mid)**2 + (cy + perp_dy - cy_mid)**2 < \
           (cx - perp_dx - cx_mid)**2 + (cy - perp_dy - cy_mid)**2:
            perp_dx, perp_dy = -perp_dx, -perp_dy

        ax.text(cx + perp_dx, cy + perp_dy, 'flow',
                ha='center', va='center', fontsize=fontsize, color=color,
                rotation=text_angle_deg, rotation_mode='anchor')

In [6]:
def get_flow_color(flow_condition: str) -> str:
    colors = {
        'laminar':   '#b83f3fff',
        'turbulent': '#bfd6e8ff',
        'stillair':  '#084a72ff',
        'still':     '#084a72ff',
        'unknown':   '#8a7d89be',
        'unknown1':  '#201c20b8',
    }
    return colors.get(flow_condition, '#000000ff')  # defaults to black if not found

In [7]:
def get_bar_plot(df,col, ax=None, ylabel=False):
    if ax==None:
        fig, ax = plt.subplots(figsize=(2,4))

    
    ax.bar(np.array([0,0.25]), df[col], align='center',width=0.15)
    
    # Iterate through bars and set color based on condition
    for i, bar in enumerate(ax.patches):
        bar.set_color( get_flow_color( df['flow_condition'].iloc[i] ) )
        bar.set_edgecolor('none')
    ax.set_ylim(0,int(np.ceil(df[col].max())))
    fifi.mpl_functions.adjust_spines(ax, ['left'], 
                                     yticks =[0,int(np.ceil(df[col].max()))], 
                                     tick_length=1.5,
                                     spine_locations={'left': 2, 'bottom': 2},
                                     linewidth=0.5)
    format_ticks_as_latex(ax)
    ax.tick_params(axis='y', pad=1)
    ax.tick_params(axis='x', pad=1)

    if ylabel:
        ax.set_ylabel(to_sentence_case(col.replace('_', ' ')), labelpad=-2)
    
    fifi.mpl_functions.set_fontsize(ax, 5)
    plt.show()
    return ax

## moth

In [8]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

ax_slope = layout.axes[('moth', 'slope')]
ax_axis_ratio = layout.axes[('moth', 'axis_ratio')]

get_bar_plot(moth, 'slope', ax_slope, ylabel=True)
get_bar_plot(moth, 'axis_ratio', ax_axis_ratio, ylabel=True)

layout.append_figure_to_layer(layout.figures['moth'], 'moth', cleartarget=True)
layout.write_svg(FIGURE_NAME)

/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


## albatross

In [9]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

ax_slope = layout.axes[('albatross', 'slope')]
ax_axis_ratio = layout.axes[('albatross', 'axis_ratio')]

get_bar_plot(albatross, 'slope', ax_slope, ylabel=False)
get_bar_plot(albatross, 'axis_ratio', ax_axis_ratio, ylabel=False)

layout.append_figure_to_layer(layout.figures['albatross'], 'albatross', cleartarget=True)
layout.write_svg(FIGURE_NAME)

## shark

In [10]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

ax_slope = layout.axes[('shark', 'slope')]
ax_axis_ratio = layout.axes[('shark', 'axis_ratio')]

get_bar_plot(shark, 'slope', ax_slope, ylabel=False)
get_bar_plot(shark, 'axis_ratio', ax_axis_ratio, ylabel=False)

layout.append_figure_to_layer(layout.figures['shark'], 'shark', cleartarget=True)
layout.write_svg(FIGURE_NAME)

# Make trajectory plots

In [11]:
def get_trajectory_filenames(animal_df):
    objids = animal_df.objid.values
    if 'albatross' not in animal_df.species.values:
        species = animal_df.species.values[0].rstrip('s')
    else:
        species = 'albatross'
    fnames = [find_file('animal_trajectories/'+species, objid, 'trajec.parquet') for objid in objids]
    print(fnames)
    dfs = [pd.read_parquet(f) for f in fnames]
    flow_conditions = animal_df.flow_condition.values
    print(flow_conditions)
    colors = [get_flow_color(fc) for fc in flow_conditions]
    return species, dfs, colors

In [12]:
fifi_labels = ['casting', 'circling']
for animal_df in [moth, albatross, shark]:

    layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
    plt.close('all')
    
    species, dfs, colors = get_trajectory_filenames(animal_df)


    for i in range(2):
        df = dfs[i]
        fifi_label = fifi_labels[i]

        ax = layout.axes[(species+'_trajec', fifi_label)]

        x = df[df.time_relative_to_flash>0].x.values
        y = df[df.time_relative_to_flash>0].y.values
        color = colors[i]

        flow_direction=df.flow_direction.values[0]
        if fifi_label == 'circling':
            flow_direction = None
        
        plot_arrowhead_trajectory_scaled(x, y, color=color, arrow_length=0.1, arrow_angle=30,
                                             ax=ax, linewidth=0.75, scale_bar=True, units='m', 
                                         flow_direction=flow_direction,
                                         flow_arrow_length=0.05, flow_arrow_angle=45,
                                        flow_arrow_size=0.15, fontsize=4, padding=0.1,
                                     flow_column_width=0.2)

        fifi.mpl_functions.adjust_spines(ax, [])

    layout.append_figure_to_layer(layout.figures[species+'_trajec'], species+'_trajec', cleartarget=True)
    layout.write_svg(FIGURE_NAME)

['animal_trajectories/moth/1Oct08B_trajec.parquet', 'animal_trajectories/moth/1Oct08E_trajec.parquet']
['laminar' 'turbulent']


/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


['animal_trajectories/albatross/albatross_casting_trajec_trajec.parquet', 'animal_trajectories/albatross/albatross_circling_trajec_trajec.parquet']
['unknown' 'unknown1']


/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


['animal_trajectories/shark/shark_d_trajec.parquet', 'animal_trajectories/shark/shark_a_trajec.parquet']
['laminar' 'still']


# Slopes vs log(speed/length)

In [13]:
df = pd.read_parquet('df_all_species.parquet')

In [14]:
df[df['species']=='RL_agents']

,axis_ratio,slope,species,objid,body_length,group,flow_speed,movement_speed,reynolds,flow_condition,distance_travelled,visual_acuity_cpd
9,0.699893,5.200235,RL_agents,rl_agent_c,NaN,aerial,0.5,2.858896,NaN,laminar,18.925888,NaN


In [15]:
df.loc[9,'body_length'] = 0.02

In [16]:
df['log(movementspeed/length)'] = np.log(df.movement_speed/df.body_length)

In [17]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

ax = layout.axes[('scaling', 'scaling')]

col='log(movementspeed/length)'


for i in range(len(df)):
    point = df.iloc[i]
    if point['flow_condition']=='laminar':
        ax.plot(point[col],point['slope'], '.',markersize=10, color='#b83f3fff')
    elif point['flow_condition']=='stillair':
        ax.plot(point[col],point['slope'], '.',markersize=10, color='#084a72ff')
    elif point['flow_condition']=='still':
        ax.plot(point[col],point['slope'], '.',markersize=10, color='#084a72ff')    
    elif point['flow_condition']=='turbulent':
        ax.plot(point[col],point['slope'], '.',markersize=10, color='#bfd6e8ff')
    elif point['flow_condition']=='unsteady':
        ax.plot(point[col],point['slope'], '.',markersize=10, color='#bfd6e8ff')
    else:
        print(point['flow_condition'],point[col])
        ax.plot(point[col],point['slope'], '.',markersize=10,  color='#8a7d89be')
        
    #ax.text(point[col], point['slope'],point['species'], 
    #       color='black', fontsize=2, ha='left', va='center')



fifi.mpl_functions.adjust_spines(ax, ['left','bottom'], yticks =[0,1,2,3,4,5,6,7], xticks=[-6,-4,-2,0,2,4,6],spine_locations={'left':10, 'bottom':15},   tick_length=3, linewidth=.5)

ax.set_ylabel('Slope (rad/s)')
ax.set_xlabel('log(Movement speed / Body length) (1/s)')
#ax.set_xlabel(col)
#ax.set_xlim(-0.01,1)
#ax.set_aspect(.15)
#fig.savefig('scatter_all2.svg', transparent=True)

ax.tick_params(axis='y', pad=2)
ax.tick_params(axis='x', pad=2)

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)

for i in range(len(df)):
    point = df.iloc[i]
    ax.text(point[col], point['slope'],point['species'], 
           color='black', fontsize=2, ha='left', va='center')

layout.append_figure_to_layer(layout.figures['scaling'], 'scaling', cleartarget=True)
layout.write_svg(FIGURE_NAME)

/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


unknown 2.668928224545568
unknown 2.0638131107879745
trail 1.4808648414136796
trail 1.3183879692784046
unknown 3.010040121255532
None -3.1859706575132605
None -4.667200473127418
